# WP2 T2.3 Unit Mapping

Owner C task: map every numeric DB attribute to a unit-of-measurement concept and write the mapping to DBRepo metadata via the REST API. True DB `BOOLEAN` columns are excluded only when DBRepo reports a boolean type. Numeric identifiers, references, categorical codes, and ordinal values are still mapped to a dimensionless/unitless concept and documented as not physical measurements.

In [5]:
import os
import requests
import pandas as pd

try:
    from dotenv import load_dotenv
    load_dotenv(override=True)
except ImportError:
    pass

DBREPO_BASE_URL = os.getenv("DBREPO_BASE_URL", "https://test.dbrepo.tuwien.ac.at").rstrip("/")
DBREPO_USER = os.getenv("DBREPO_USER")
DBREPO_PASSWORD = os.getenv("DBREPO_PASSWORD")
DBREPO_DB_ID = os.getenv("DBREPO_DB_ID", "3d81c073-e5fd-49b9-9536-b75ed490ca3e")
API_BASE = f"{DBREPO_BASE_URL}/api/v1"

if not DBREPO_USER or not DBREPO_PASSWORD:
    raise RuntimeError("Set DBREPO_USER and DBREPO_PASSWORD as environment variables or in a local .env file.")

AUTH = (DBREPO_USER, DBREPO_PASSWORD)
PROJECT_TABLES = {"collision", "vehicle", "casualty"}

print(f"DBRepo API: {API_BASE}")
print(f"Database: {DBREPO_DB_ID}")

DBRepo API: https://test.dbrepo.tuwien.ac.at/api/v1
Database: 3d81c073-e5fd-49b9-9536-b75ed490ca3e


## Expected Unit Decisions

The SI Digital Framework is used first. OMG Commons Quantities and Units is the assignment-defined fallback. QUDT is used only where a concrete unit individual is needed and the first two sources do not provide one.

In [6]:
SI_ONE = "https://si-digital-framework.org/SI/units/one"
SI_METRE = "https://si-digital-framework.org/SI/units/metre"
SI_DEGREE = "https://si-digital-framework.org/SI/units/degree"
QUDT_MPH = "http://qudt.org/vocab/unit/MI-PER-HR"
QUDT_YEAR = "http://qudt.org/vocab/unit/YR"
QUDT_CUBIC_CENTIMETRE = "http://qudt.org/vocab/unit/CentiM3"

UNIT_MAPPINGS = {
    "collision": {
        "collision_year": ("numeric code", "one / dimensionless", SI_ONE, "SI Digital Framework", "Calendar year value; not a physical measurement in this schema."),
        "location_easting_osgr": ("measurement", "metre", SI_METRE, "SI Digital Framework", "British National Grid easting coordinate in metres."),
        "location_northing_osgr": ("measurement", "metre", SI_METRE, "SI Digital Framework", "British National Grid northing coordinate in metres."),
        "longitude": ("measurement", "degree", SI_DEGREE, "SI Digital Framework", "Geographic longitude in decimal degrees."),
        "latitude": ("measurement", "degree", SI_DEGREE, "SI Digital Framework", "Geographic latitude in decimal degrees."),
        "number_of_vehicles": ("count", "one / dimensionless", SI_ONE, "SI Digital Framework", "Count of vehicles involved."),
        "number_of_casualties": ("count", "one / dimensionless", SI_ONE, "SI Digital Framework", "Count of casualties involved."),
        "first_road_number": ("numeric code", "one / dimensionless", SI_ONE, "SI Digital Framework", "Road route number, not a measurement."),
        "speed_limit": ("measurement", "mile per hour", QUDT_MPH, "other justified ontology", "UK STATS19 speed limits are recorded in miles per hour; SI/OMG Commons do not provide this concrete unit individual."),
        "second_road_number": ("numeric code", "one / dimensionless", SI_ONE, "SI Digital Framework", "Road route number, not a measurement."),
        "did_police_officer_attend_scene_of_accident": ("numeric flag", "one / dimensionless", SI_ONE, "SI Digital Framework", "Mapped only if DBRepo stores the flag as numeric rather than true BOOLEAN."),
        "trunk_road_flag": ("numeric flag", "one / dimensionless", SI_ONE, "SI Digital Framework", "Mapped only if DBRepo stores the flag as numeric rather than true BOOLEAN."),
        "collision_injury_based": ("numeric flag", "one / dimensionless", SI_ONE, "SI Digital Framework", "Mapped only if DBRepo stores the flag as numeric rather than true BOOLEAN."),
        "collision_adjusted_severity_serious": ("numeric flag", "one / dimensionless", SI_ONE, "SI Digital Framework", "Mapped only if DBRepo stores the flag as numeric rather than true BOOLEAN."),
        "collision_adjusted_severity_slight": ("numeric flag", "one / dimensionless", SI_ONE, "SI Digital Framework", "Mapped only if DBRepo stores the flag as numeric rather than true BOOLEAN."),
    },
    "vehicle": {
        "vehicle_id": ("identifier", "one / dimensionless", SI_ONE, "SI Digital Framework", "Surrogate identifier, not a physical measurement."),
        "vehicle_reference": ("reference", "one / dimensionless", SI_ONE, "SI Digital Framework", "Source reference number within a collision."),
        "vehicle_left_hand_drive": ("numeric flag", "one / dimensionless", SI_ONE, "SI Digital Framework", "Mapped only if DBRepo stores the flag as numeric rather than true BOOLEAN."),
        "age_of_driver": ("measurement", "year", QUDT_YEAR, "other justified ontology", "Age is expressed in years; SI/OMG Commons do not provide a concrete year unit individual."),
        "engine_capacity_cc": ("measurement", "cubic centimetre", QUDT_CUBIC_CENTIMETRE, "other justified ontology", "Engine capacity is recorded in cubic centimetres; SI/OMG Commons do not provide this concrete unit individual."),
        "age_of_vehicle": ("measurement", "year", QUDT_YEAR, "other justified ontology", "Vehicle age is expressed in years; SI/OMG Commons do not provide a concrete year unit individual."),
        "driver_imd_decile": ("ordinal code", "one / dimensionless", SI_ONE, "SI Digital Framework", "Decile code, not a physical measurement."),
        "escooter_flag": ("numeric flag", "one / dimensionless", SI_ONE, "SI Digital Framework", "Mapped only if DBRepo stores the flag as numeric rather than true BOOLEAN."),
    },
    "casualty": {
        "casualty_id": ("identifier", "one / dimensionless", SI_ONE, "SI Digital Framework", "Surrogate identifier, not a physical measurement."),
        "vehicle_reference": ("reference", "one / dimensionless", SI_ONE, "SI Digital Framework", "Source reference number linking a casualty to a vehicle."),
        "casualty_reference": ("reference", "one / dimensionless", SI_ONE, "SI Digital Framework", "Source casualty reference number within a collision."),
        "age_of_casualty": ("measurement", "year", QUDT_YEAR, "other justified ontology", "Casualty age is expressed in years; SI/OMG Commons do not provide a concrete year unit individual."),
        "casualty_imd_decile": ("ordinal code", "one / dimensionless", SI_ONE, "SI Digital Framework", "Decile code, not a physical measurement."),
        "casualty_injury_based": ("numeric flag", "one / dimensionless", SI_ONE, "SI Digital Framework", "Mapped only if DBRepo stores the flag as numeric rather than true BOOLEAN."),
        "casualty_adjusted_severity_serious": ("numeric flag", "one / dimensionless", SI_ONE, "SI Digital Framework", "Mapped only if DBRepo stores the flag as numeric rather than true BOOLEAN."),
        "casualty_adjusted_severity_slight": ("numeric flag", "one / dimensionless", SI_ONE, "SI Digital Framework", "Mapped only if DBRepo stores the flag as numeric rather than true BOOLEAN."),
    },
}

mapping_rows = []
for table_name, columns in UNIT_MAPPINGS.items():
    for column_name, (numeric_type, label, uri, source, reason) in columns.items():
        mapping_rows.append({
            "table": table_name,
            "column": column_name,
            "numeric_type": numeric_type,
            "selected_unit_label": label,
            "selected_unit_uri": uri,
            "ontology_source": source,
            "reason": reason,
        })

expected_mapping_df = pd.DataFrame(mapping_rows)
display(expected_mapping_df)

,table,column,numeric_type,selected_unit_label,selected_unit_uri,ontology_source,reason
0,collision,collision_year,numeric code,one / dimensionless,https://si-digital-framework.org/SI/units/one,SI Digital Framework,Calendar year value; not a physical measuremen...
1,collision,location_easting_osgr,measurement,metre,https://si-digital-framework.org/SI/units/metre,SI Digital Framework,British National Grid easting coordinate in me...
2,collision,location_northing_osgr,measurement,metre,https://si-digital-framework.org/SI/units/metre,SI Digital Framework,British National Grid northing coordinate in m...
3,collision,longitude,measurement,degree,https://si-digital-framework.org/SI/units/degree,SI Digital Framework,Geographic longitude in decimal degrees.
4,collision,latitude,measurement,degree,https://si-digital-framework.org/SI/units/degree,SI Digital Framework,Geographic latitude in decimal degrees.
5,collision,number_of_vehicles,count,one / dimensionless,https://si-digital-framework.org/SI/units/one,SI Digital Framework,Count of vehicles involved.
6,collision,number_of_casualties,count,one / dimensionless,https://si-digital-framework.org/SI/units/one,SI Digital Framework,Count of casualties involved.
7,collision,first_road_number,numeric code,one / dimensionless,https://si-digital-framework.org/SI/units/one,SI Digital Framework,"Road route number, not a measurement."
8,collision,speed_limit,measurement,mile per hour,http://qudt.org/vocab/unit/MI-PER-HR,other justified ontology,UK STATS19 speed limits are recorded in miles ...
9,collision,second_road_number,numeric code,one / dimensionless,https://si-digital-framework.org/SI/units/one,SI Digital Framework,"Road route number, not a measurement."


## Fetch DBRepo Metadata

In [7]:
NUMERIC_TYPES = {
    "serial", "tinyint", "smallint", "mediumint", "int", "integer", "bigint",
    "float", "double", "decimal", "numeric", "real", "year"
}
BOOLEAN_TYPES = {"bool", "boolean"}

def request_json(method, path, **kwargs):
    url = f"{API_BASE}{path}"
    response = requests.request(method, url, auth=AUTH, timeout=60, **kwargs)
    if response.status_code >= 400:
        raise RuntimeError(f"{method} {url} failed: {response.status_code} {response.text}")
    if not response.text:
        return None
    return response.json()

def normalized_type(column):
    return str(column.get("type", "")).lower()

def is_true_boolean(column):
    return normalized_type(column) in BOOLEAN_TYPES

def is_numeric(column):
    return normalized_type(column) in NUMERIC_TYPES

def fetch_project_metadata():
    table_summaries = request_json("GET", f"/database/{DBREPO_DB_ID}/table")
    tables = {}
    for table_summary in table_summaries:
        table_name = table_summary.get("name")
        if table_name not in PROJECT_TABLES:
            continue
        table_id = table_summary.get("id")
        detail = request_json("GET", f"/database/{DBREPO_DB_ID}/table/{table_id}")
        tables[table_name] = detail
    missing = PROJECT_TABLES - set(tables)
    if missing:
        raise RuntimeError(f"Missing DBRepo tables: {sorted(missing)}")
    return tables

tables = fetch_project_metadata()
print("Fetched tables:", ", ".join(sorted(tables)))

Fetched tables: casualty, collision, vehicle


In [8]:
numeric_rows = []
excluded_boolean_rows = []

for table_name, table in tables.items():
    for column in table.get("columns", []):
        column_name = column.get("name")
        column_type = normalized_type(column)
        if is_true_boolean(column):
            excluded_boolean_rows.append({
                "table": table_name,
                "column": column_name,
                "dbrepo_type": column_type,
                "reason": "Excluded because DBRepo reports a true boolean type.",
            })
            continue
        if is_numeric(column):
            decision = UNIT_MAPPINGS.get(table_name, {}).get(column_name)
            if decision is None:
                raise RuntimeError(f"Numeric column missing unit mapping: {table_name}.{column_name} ({column_type})")
            numeric_type, label, uri, source, reason = decision
            numeric_rows.append({
                "table": table_name,
                "table_id": table.get("id"),
                "column": column_name,
                "column_id": column.get("id"),
                "dbrepo_type": column_type,
                "numeric_type": numeric_type,
                "selected_unit_label": label,
                "expected_unit_uri": uri,
                "ontology_source": source,
                "reason": reason,
                "current_unit_uri": column.get("unit_uri"),
            })

numeric_columns_df = pd.DataFrame(numeric_rows).sort_values(["table", "column"])
excluded_booleans_df = pd.DataFrame(
    excluded_boolean_rows,
    columns=["table", "column", "dbrepo_type", "reason"],
).sort_values(["table", "column"])

print(f"Numeric DB columns to map: {len(numeric_columns_df)}")
display(numeric_columns_df[["table", "column", "dbrepo_type", "numeric_type", "selected_unit_label", "expected_unit_uri", "ontology_source", "reason"]])

print(f"True BOOLEAN columns excluded: {len(excluded_booleans_df)}")
display(excluded_booleans_df)

Numeric DB columns to map: 31


,table,column,dbrepo_type,numeric_type,selected_unit_label,expected_unit_uri,ontology_source,reason
26,casualty,age_of_casualty,double,measurement,year,http://qudt.org/vocab/unit/YR,other justified ontology,Casualty age is expressed in years; SI/OMG Com...
29,casualty,casualty_adjusted_severity_serious,double,numeric flag,one / dimensionless,https://si-digital-framework.org/SI/units/one,SI Digital Framework,Mapped only if DBRepo stores the flag as numer...
30,casualty,casualty_adjusted_severity_slight,double,numeric flag,one / dimensionless,https://si-digital-framework.org/SI/units/one,SI Digital Framework,Mapped only if DBRepo stores the flag as numer...
23,casualty,casualty_id,double,identifier,one / dimensionless,https://si-digital-framework.org/SI/units/one,SI Digital Framework,"Surrogate identifier, not a physical measurement."
27,casualty,casualty_imd_decile,double,ordinal code,one / dimensionless,https://si-digital-framework.org/SI/units/one,SI Digital Framework,"Decile code, not a physical measurement."
28,casualty,casualty_injury_based,double,numeric flag,one / dimensionless,https://si-digital-framework.org/SI/units/one,SI Digital Framework,Mapped only if DBRepo stores the flag as numer...
25,casualty,casualty_reference,double,reference,one / dimensionless,https://si-digital-framework.org/SI/units/one,SI Digital Framework,Source casualty reference number within a coll...
24,casualty,vehicle_reference,double,reference,one / dimensionless,https://si-digital-framework.org/SI/units/one,SI Digital Framework,Source reference number linking a casualty to ...
21,collision,collision_adjusted_severity_serious,double,numeric flag,one / dimensionless,https://si-digital-framework.org/SI/units/one,SI Digital Framework,Mapped only if DBRepo stores the flag as numer...
22,collision,collision_adjusted_severity_slight,double,numeric flag,one / dimensionless,https://si-digital-framework.org/SI/units/one,SI Digital Framework,Mapped only if DBRepo stores the flag as numer...


True BOOLEAN columns excluded: 0


,table,column,dbrepo_type,reason


## Write Unit URIs to DBRepo

In [9]:
APPLY_DBREPO_UPDATES = os.getenv("APPLY_DBREPO_UPDATES", "true").lower() == "true"

update_rows = []
for _, row in numeric_columns_df.iterrows():
    path = f"/database/{DBREPO_DB_ID}/table/{row['table_id']}/column/{row['column_id']}"
    payload = {"unit_uri": row["expected_unit_uri"]}
    if APPLY_DBREPO_UPDATES:
        request_json("PUT", path, json=payload, headers={"Accept": "application/json", "Content-Type": "application/json"})
        status = "updated"
    else:
        status = "dry_run"
    update_rows.append({
        "table": row["table"],
        "column": row["column"],
        "expected_unit_uri": row["expected_unit_uri"],
        "status": status,
    })

unit_update_report = pd.DataFrame(update_rows)
display(unit_update_report)
print(unit_update_report["status"].value_counts().to_string())

,table,column,expected_unit_uri,status
0,casualty,age_of_casualty,http://qudt.org/vocab/unit/YR,updated
1,casualty,casualty_adjusted_severity_serious,https://si-digital-framework.org/SI/units/one,updated
2,casualty,casualty_adjusted_severity_slight,https://si-digital-framework.org/SI/units/one,updated
3,casualty,casualty_id,https://si-digital-framework.org/SI/units/one,updated
4,casualty,casualty_imd_decile,https://si-digital-framework.org/SI/units/one,updated
5,casualty,casualty_injury_based,https://si-digital-framework.org/SI/units/one,updated
6,casualty,casualty_reference,https://si-digital-framework.org/SI/units/one,updated
7,casualty,vehicle_reference,https://si-digital-framework.org/SI/units/one,updated
8,collision,collision_adjusted_severity_serious,https://si-digital-framework.org/SI/units/one,updated
9,collision,collision_adjusted_severity_slight,https://si-digital-framework.org/SI/units/one,updated


status
updated    31


## Read Back and Verify DBRepo Metadata

In [10]:
verified_tables = fetch_project_metadata()
verification_rows = []

for table_name, table in verified_tables.items():
    for column in table.get("columns", []):
        if is_true_boolean(column) or not is_numeric(column):
            continue
        column_name = column.get("name")
        expected = UNIT_MAPPINGS[table_name][column_name][2]
        actual = column.get("unit_uri")
        verification_rows.append({
            "table": table_name,
            "column": column_name,
            "dbrepo_type": normalized_type(column),
            "expected_unit_uri": expected,
            "actual_unit_uri": actual,
            "matches": expected == actual,
        })

unit_mapping_verification = pd.DataFrame(verification_rows).sort_values(["table", "column"])
display(unit_mapping_verification)

matching = int(unit_mapping_verification["matches"].sum())
total = len(unit_mapping_verification)
print(f"Matching unit URIs: {matching} / {total}")

if matching != total:
    mismatches = unit_mapping_verification.loc[~unit_mapping_verification["matches"]]
    raise AssertionError("Some DBRepo unit_uri values do not match the expected mapping. See mismatches above.")

,table,column,dbrepo_type,expected_unit_uri,actual_unit_uri,matches
26,casualty,age_of_casualty,double,http://qudt.org/vocab/unit/YR,http://qudt.org/vocab/unit/YR,True
29,casualty,casualty_adjusted_severity_serious,double,https://si-digital-framework.org/SI/units/one,https://si-digital-framework.org/SI/units/one,True
30,casualty,casualty_adjusted_severity_slight,double,https://si-digital-framework.org/SI/units/one,https://si-digital-framework.org/SI/units/one,True
23,casualty,casualty_id,double,https://si-digital-framework.org/SI/units/one,https://si-digital-framework.org/SI/units/one,True
27,casualty,casualty_imd_decile,double,https://si-digital-framework.org/SI/units/one,https://si-digital-framework.org/SI/units/one,True
28,casualty,casualty_injury_based,double,https://si-digital-framework.org/SI/units/one,https://si-digital-framework.org/SI/units/one,True
25,casualty,casualty_reference,double,https://si-digital-framework.org/SI/units/one,https://si-digital-framework.org/SI/units/one,True
24,casualty,vehicle_reference,double,https://si-digital-framework.org/SI/units/one,https://si-digital-framework.org/SI/units/one,True
21,collision,collision_adjusted_severity_serious,double,https://si-digital-framework.org/SI/units/one,https://si-digital-framework.org/SI/units/one,True
22,collision,collision_adjusted_severity_slight,double,https://si-digital-framework.org/SI/units/one,https://si-digital-framework.org/SI/units/one,True


Matching unit URIs: 31 / 31
